# BASELINE — Gemma 4 ZERO-SHOT (TRƯỚC fine-tune)

Chạy Gemma 4 **gốc** (chưa train) trên ĐÚNG test set → JSON `gemma4_zeroshot` để so **before/after** với gemma4_v3.
**KHÔNG train, KHÔNG load checkpoint.** Run-All trên A100 (~15-20 phút).
Cùng split/prompt/parser với bản fine-tune → so được.

In [ ]:
# === Cài thư viện thiếu trên Colab (chạy 1 lần đầu session) ===
# bitsandbytes: nén 4-bit | accelerate: đặt model lên GPU (device_map="auto")
%pip install -q -U bitsandbytes accelerate peft torchao


In [ ]:
import os, json, re, random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("Import xong | CUDA:", torch.cuda.is_available())

In [ ]:
MODEL_ID = "google/gemma-4-E4B-it"    # thay gemma-3n-e4b-it
USE_MOCK    = False           # False: chạy model thật (cần GPU); True: chạy thử không cần model
DATA_DIR    = None            # gán ở cell dữ liệu = data_liver
COORD_SCALE = 1000            # toạ độ box [0,1000)
N_SAMPLES   = 5               # số lần sample đo nhất quán không gian (Hướng 1)
IOU_CORRECT = 0.5             # ngưỡng IoU coi là đúng
EVAL_LIMIT  = None              # số ảnh test chạy ở phần đánh giá (None = hết)
MAX_NEW_TOKENS = 192           # số token tối đa model sinh (box ngắn, 64 đủ) — dùng ở cell kiểm tra cuối
SEED        = 0
RUN_NAME    = "gemma4_zeroshot"   # BASELINE zero-shot (truoc train)      # <<< TÊN LẦN TRAIN — ĐỔI mỗi lần train MỚI để không ghi đè checkpoint cũ

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("DEVICE:", DEVICE, "| USE_MOCK:", USE_MOCK)

LOAD_IN_4BIT = True   # notebook train -> luon 4-bit (QLoRA)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Ví dụ đường dẫn lưu/đọc data trên Drive:
import os
DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks/medrega/data"   # đổi theo ý mày
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Drive đã mount. Lưu data tại:", DRIVE_DIR)

In [ ]:
# === Lấy data_liver_multi (1238 mẫu) từ Drive ===
import os, zipfile, glob
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/medrega/data_multi_v3/data_liver"
zip_path = "/content/drive/MyDrive/Colab Notebooks/medrega/data_multi_v3/data_liver_multi_v3.zip"

if not os.path.exists(os.path.join(DATA_DIR, "data.jsonl")):
    os.makedirs(DATA_DIR, exist_ok=True)
    print("Giải nén ->", DATA_DIR)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)
else:
    print("Đã có data_multi -> dùng luôn.")

n_row = sum(1 for _ in open(os.path.join(DATA_DIR, "data.jsonl"), encoding="utf-8"))
print("DATA_DIR =", DATA_DIR, "| jsonl:", n_row)   # phải in 1238

In [ ]:
# === Đọc data.jsonl + chia train/cal/test THEO BỆNH NHÂN (không lộ dữ liệu) ===
# train: để fine-tune | cal: hiệu chỉnh ngưỡng (Hướng 1) | test: đánh giá cuối.
import json, numpy as np

rows = [json.loads(l) for l in open(os.path.join(DATA_DIR, "data.jsonl"), encoding="utf-8")]

rng = np.random.default_rng(SEED)                    # SEED từ cell config (Section 2)
pids = sorted({r["patient_id"] for r in rows}); rng.shuffle(pids)
n = len(pids)
n_tr, n_ca = int(n * 0.6), int(n * 0.2)              # 60% train / 20% cal / 20% test
tr = set(pids[:n_tr]); ca = set(pids[n_tr:n_tr + n_ca])
dataset = {
    "train": [r for r in rows if r["patient_id"] in tr],
    "cal":   [r for r in rows if r["patient_id"] in ca],
    "test":  [r for r in rows if r["patient_id"] not in tr and r["patient_id"] not in ca],
}
OUT_DIR = DATA_DIR

# kiểm tra không lộ dữ liệu: 3 tập không trùng bệnh nhân
_s = {k: {r["patient_id"] for r in v} for k, v in dataset.items()}
assert not (_s["train"] & _s["cal"]) and not (_s["train"] & _s["test"]) and not (_s["cal"] & _s["test"])
print("Tổng mẫu:", len(rows),
      "| dương:", sum(r["label"] == "tumor" for r in rows),
      "| âm:", sum(r["label"] == "none" for r in rows))
print("Split (theo bệnh nhân):", {k: len(v) for k, v in dataset.items()})
print("Bệnh nhân:", {k: len(s) for k, s in _s.items()}, "-> không trùng (OK)")

## Tải model GỐC (BF16, không train, không checkpoint)

In [ ]:
# === Tải model Gemma BF16 + processor (A100, KHÔNG 4-bit) ===
import os, torch
MEDREGA_ROOT = os.path.dirname(DRIVE_DIR.rstrip("/"))
HF_CACHE = "/content/hf_cache"
CKPT_DIR = os.path.join(MEDREGA_ROOT, "model", "checkpoints")
os.makedirs(HF_CACHE, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)

def _hf_token():
    try:
        from google.colab import userdata
        if userdata.get("HF_TOKEN"): return userdata.get("HF_TOKEN")
    except Exception:
        pass
    env = os.path.join(MEDREGA_ROOT, "secrets.env")
    if os.path.exists(env):
        for line in open(env, encoding="utf-8"):
            line = line.strip()
            if line.startswith("HF_TOKEN") and "=" in line:
                return line.split("=", 1)[1].strip().strip(chr(34)).strip(chr(39))
    return os.environ.get("HF_TOKEN")

from huggingface_hub import login
login(token=_hf_token())

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=HF_CACHE)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, cache_dir=HF_CACHE, torch_dtype=torch.bfloat16, device_map={"": 0})
model.eval()
print("Đã tải model BF16:", MODEL_ID, "| VRAM:", round(torch.cuda.memory_allocated()/1e9, 1), "GB")

In [ ]:
# Định dạng record -> input Gemma + AUGMENT (on-the-fly, CHỈ train) + build_train_example | MULTI-BOX
from PIL import Image
import os, math, random as _rnd
import numpy as np

INSTRUCT_PROMPT_TRAIN = """You are a medical imaging assistant. This abdominal CT image may or may not contain liver tumors (abnormal masses or lesions inside the liver). There may be ONE, SEVERAL, or NO tumors. Examine the liver region carefully.
Coordinate system: treat the image as a 1000x1000 grid. (0,0) is the TOP-LEFT corner and (1000,1000) is the BOTTOM-RIGHT corner. All coordinates are integers from 0 to 1000.
Respond with EXACTLY ONE of these two options and nothing else:
1. If one or more tumors are present, give a bounding box for EACH tumor: <ref>liver tumor</ref><box>[[x1, y1, x2, y2], [x1, y1, x2, y2], ...]</box>
2. If there is no tumor: No liver tumor is found."""

PROMPT_2OPT = INSTRUCT_PROMPT_TRAIN
PROMPT_3OPT = INSTRUCT_PROMPT_TRAIN.replace(
    "Respond with EXACTLY ONE of these two options and nothing else:",
    "Respond with EXACTLY ONE of these three options and nothing else:") + \
    "\n3. If you are genuinely UNSURE whether a tumor is present: Uncertain."
INSTRUCT_PROMPT_TEST = PROMPT_2OPT

AUGMENT = True          # bật augment khi train (val/test gọi với do_augment=False)

def load_image(rec, root=None):
    root = root or globals().get("DATA_DIR") or globals().get("OUT_DIR") or "."
    path = rec["image_path"]
    if not os.path.isabs(path):
        path = os.path.join(root, path)
    return Image.open(path).convert("RGB")

def augment(img, boxes, p=0.9):
    """MULTI-BOX. boxes: list [x1,y1,x2,y2] thang [0,COORD_SCALE) (rỗng = ca âm).
       Áp CÙNG affine cho mọi box; bỏ box ra ngoài khung. Hết box -> giữ ảnh+boxes GỐC.
       Dịch ±18% / xoay ±12° / scale ±10% / sáng nhẹ. KHÔNG flip/xoay90 (CT gan)."""
    if _rnd.random() > p:
        return img, boxes
    W, H = img.size
    angle = _rnd.uniform(-12, 12); scale = _rnd.uniform(0.9, 1.1)
    dx = _rnd.uniform(-0.18, 0.18) * W; dy = _rnd.uniform(-0.18, 0.18) * H
    cx, cy = W / 2.0, H / 2.0
    a = math.radians(angle); cos, sin = math.cos(a), math.sin(a)
    ia, ib = cos / scale, sin / scale
    ic = -ia * (cx + dx) - ib * (cy + dy) + cx
    id_, ie = -sin / scale, cos / scale
    if_ = -id_ * (cx + dx) - ie * (cy + dy) + cy
    img2 = img.transform((W, H), Image.AFFINE, (ia, ib, ic, id_, ie, if_),
                         resample=Image.BILINEAR, fillcolor=(0, 0, 0))
    arr = np.asarray(img2).astype(np.float32) * _rnd.uniform(0.85, 1.15)
    img2 = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    if not boxes:
        return img2, boxes
    def fwd(x, y):
        return (scale * (cos * (x - cx) - sin * (y - cy)) + cx + dx,
                scale * (sin * (x - cx) + cos * (y - cy)) + cy + dy)
    new_boxes = []
    for box in boxes:
        px = [box[0] / COORD_SCALE * W, box[2] / COORD_SCALE * W]
        py = [box[1] / COORD_SCALE * H, box[3] / COORD_SCALE * H]
        cs = [fwd(x, y) for x in px for y in py]
        xs = [c[0] for c in cs]; ys = [c[1] for c in cs]
        nx1, ny1, nx2, ny2 = max(0, min(xs)), max(0, min(ys)), min(W, max(xs)), min(H, max(ys))
        if nx2 - nx1 < 4 or ny2 - ny1 < 4:
            continue                                   # box này ra ngoài -> bỏ riêng box này
        new_boxes.append([int(nx1 / W * COORD_SCALE), int(ny1 / H * COORD_SCALE),
                          int(nx2 / W * COORD_SCALE), int(ny2 / H * COORD_SCALE)])
    if not new_boxes:
        return img, boxes                              # tất cả ra ngoài -> giữ GỐC (không đổi nhãn)
    return img2, new_boxes

def to_messages(rec, with_answer, prompt):
    msgs = [{"role": "user", "content": [
        {"type": "image", "image": load_image(rec)},
        {"type": "text",  "text": prompt}]}]
    if with_answer:
        msgs.append({"role": "model",
                     "content": [{"type": "text", "text": rec["conversations"][-1]["value"]}]})
    return msgs

def build_infer_inputs(rec, processor, device=None, prompt=None):
    """INFERENCE (eval): KHÔNG augment. prompt riêng cho conf_self."""
    inputs = processor.apply_chat_template(
        to_messages(rec, False, prompt or INSTRUCT_PROMPT_TEST),
        add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    return inputs.to(device) if device else inputs

def _answer_from_boxes(boxes):
    if not boxes:
        return "No liver tumor is found."
    inner = ", ".join(f"[{b[0]}, {b[1]}, {b[2]}, {b[3]}]" for b in boxes)
    return f"<ref>liver tumor</ref><box>[{inner}]</box>"

def build_train_example(rec, processor, do_augment=None):
    """TRAIN multi-box: AUGMENT ảnh+boxes on-the-fly. do_augment=None -> AUGMENT global (train=True, val=False).
       Loss CHỈ trên token đáp án. _is_pos cho W_POS weighting."""
    import torch
    do_augment = AUGMENT if do_augment is None else do_augment
    img = load_image(rec); boxes = list(rec.get("gt_boxes") or [])
    if do_augment:
        img, boxes = augment(img, boxes)
    ans = _answer_from_boxes(boxes)
    msgs_full = [{"role": "user", "content": [{"type": "image", "image": img},
                  {"type": "text", "text": INSTRUCT_PROMPT_TRAIN}]},
                 {"role": "model", "content": [{"type": "text", "text": ans}]}]
    full = processor.apply_chat_template(msgs_full, add_generation_prompt=False,
                  tokenize=True, return_dict=True, return_tensors="pt")
    prompt = processor.apply_chat_template(msgs_full[:1], add_generation_prompt=True,
                  tokenize=True, return_dict=True, return_tensors="pt")
    labels = full["input_ids"].clone()
    labels[:, :prompt["input_ids"].shape[1]] = -100
    out = dict(full); out["labels"] = labels
    out["_is_pos"] = torch.tensor([1.0 if boxes else 0.0])
    return out

print("Format MULTI-BOX + AUGMENT (train) sẵn sàng. AUGMENT =", AUGMENT)

In [ ]:
# === Hàm tiện ích — parse_box + IoU (cho cell kiểm tra cuối) ===
import re

def parse_box(text):
    """Đọc box từ text model -> [x1,y1,x2,y2]. None nếu model nói không có u / không có số.
       Xử lý: <box>[[x1,y1,x2,y2]]</box>, json {x,y,width,height} (xywh), hoặc 4 số đầu tiên."""
    t = str(text)
    # 1) model nói KHÔNG có u (và không kèm toạ độ) -> None (đúng cho ca âm)
    neg = re.search(r"no\b.{0,30}tumor|not\s.{0,20}found|cannot provide|no .{0,20}lesion", t, re.I)
    if neg and not re.search(r'"width"|"x"\s*:|<box>', t, re.I):
        return None
    # 2) json xywh: {"x":..,"y":..,"width":..,"height":..} -> đổi sang xyxy
    m = re.search(r'"x"\s*:\s*(\d+).*?"y"\s*:\s*(\d+).*?"width"\s*:\s*(\d+).*?"height"\s*:\s*(\d+)', t, re.S | re.I)
    if m:
        x, y, w, h = map(int, m.groups())
        return [x, y, x + w, y + h]
    # 3) lấy 4 số nguyên đầu tiên (bắt cả <box>[[...]]). Giả định x1,y1,x2,y2.
    nums = re.findall(r"\d+", t)
    if len(nums) >= 4:
        x1, y1, x2, y2 = map(int, nums[:4])
        if x2 > x1 and y2 > y1:
            return [x1, y1, x2, y2]
        return [x1, y1, x1 + abs(x2), y1 + abs(y2)]   # phòng khi lỡ là xywh
    return None

def iou(b1, b2):
    """IoU giữa 2 box [x1,y1,x2,y2]. None -> 0.0."""
    if b1 is None or b2 is None:
        return 0.0
    xa, ya = max(b1[0], b2[0]), max(b1[1], b2[1])
    xb, yb = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, xb - xa) * max(0, yb - ya)
    a1 = max(0, b1[2] - b1[0]) * max(0, b1[3] - b1[1])
    a2 = max(0, b2[2] - b2[0]) * max(0, b2[3] - b2[1])
    u = a1 + a2 - inter
    return inter / u if u > 0 else 0.0

# test nhanh
assert parse_box("<ref>liver tumor</ref><box>[[10, 20, 110, 220]]</box>") == [10, 20, 110, 220]
assert parse_box('{"x": 150, "y": 150, "width": 120, "height": 100}') == [150, 150, 270, 250]
assert parse_box("No liver tumor is found.") is None
assert abs(iou([0, 0, 100, 100], [50, 50, 150, 150]) - 1/7) < 1e-6
print("parse_box + iou OK (đã test xywh + <box> + ca âm)")

# === MULTI-BOX utils (Phase: multi-box) ===
import numpy as np
from scipy.optimize import linear_sum_assignment

def parse_boxes(text):
    """Đọc TẤT CẢ box từ text model -> list [x1,y1,x2,y2]. [] nếu model nói không u / không số."""
    t = str(text)
    neg = re.search(r"no\b.{0,30}tumor|not\s.{0,20}found|no .{0,20}lesion", t, re.I)
    if neg and "[" not in t:
        return []
    boxes = []
    for m in re.findall(r"\[\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\]", t):
        x1, y1, x2, y2 = map(int, m)
        if x2 > x1 and y2 > y1:
            boxes.append([x1, y1, x2, y2])
    return boxes

def match_boxes(preds, gts):
    """Hungarian match preds<->gts theo IoU (không theo thứ tự). Trả (matched_ious, n_pred, n_gt)."""
    if not preds or not gts:
        return [], len(preds), len(gts)
    M = np.zeros((len(preds), len(gts)))
    for i, p in enumerate(preds):
        for j, gv in enumerate(gts):
            M[i, j] = iou(p, gv)
    ri, ci = linear_sum_assignment(1 - M)
    return [float(M[r, c]) for r, c in zip(ri, ci)], len(preds), len(gts)

def box_metrics(preds, gts, thr=0.25):
    """Precision/recall/f1 + mean matched-IoU cho 1 lát (multi-box)."""
    mi, npd, ngt = match_boxes(preds, gts)
    hit = sum(v >= thr for v in mi)
    prec = hit / npd if npd else 0.0
    rec = hit / ngt if ngt else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    miou = float(np.mean(mi)) if mi else 0.0
    return {"prec": prec, "rec": rec, "f1": f1, "miou": miou, "n_pred": npd, "n_gt": ngt}

assert parse_boxes("<ref>liver tumor</ref><box>[[10, 20, 110, 220], [30, 40, 130, 240]]</box>") == [[10,20,110,220],[30,40,130,240]]
assert parse_boxes("No liver tumor is found.") == []
_mi, _np, _ng = match_boxes([[0,0,100,100]], [[50,50,150,150],[0,0,100,100]])
assert _np == 1 and _ng == 2 and max(_mi) == 1.0
print("parse_boxes + match_boxes (Hungarian) OK")


In [ ]:
# === [EVAL] Hàm predict MULTI-BOX + 3 tín hiệu tin cậy ===
import numpy as np, re

USE_MOCK = globals().get("USE_MOCK", False)

if USE_MOCK:
    import hashlib
    def _diff(rec):
        return (int(hashlib.md5(str(rec["id"]).encode()).hexdigest(), 16) % 1000) / 1000.0
    def predict_boxes(rec, temperature=0.0):
        gts = rec.get("gt_boxes") or []
        d = _diff(rec); jit = COORD_SCALE * (0.02 + 0.3 * d) * (1.0 if temperature > 0 else 0.4)
        out = []
        for gv in gts:
            nb = [int(np.clip(gv[i] + np.random.normal(0, jit), 0, COORD_SCALE - 1)) for i in range(4)]
            out.append([min(nb[0], nb[2]), min(nb[1], nb[3]), max(nb[0], nb[2]), max(nb[1], nb[3])])
        return out, float(np.clip(-0.1 - 2.5 * d, -5, 0))
    def conf_self(rec):
        return float(np.clip(1 - _diff(rec), 0, 1))
else:
    import torch, torch.nn.functional as F
    @torch.no_grad()
    def _generate(rec, temperature=0.0, prompt=None, max_new=None):
        inp = build_infer_inputs(rec, processor, device=model.device, prompt=prompt)
        gen = model.generate(**inp, max_new_tokens=max_new or MAX_NEW_TOKENS,
                             do_sample=temperature > 0, temperature=max(temperature, 1e-5),
                             output_scores=True, return_dict_in_generate=True)
        new = gen.sequences[0][inp["input_ids"].shape[1]:]
        return new, gen.scores
    def _coord_logprob(new, scores):
        """TB logprob CHỈ trên token là CHỮ SỐ (toạ độ) — bỏ token khuôn mẫu (fix bug -0.865)."""
        lps = []
        for t, sc in zip(new, scores):
            tok = processor.decode([int(t)], skip_special_tokens=True)
            if tok.strip().isdigit():
                lps.append(F.log_softmax(sc[0], -1)[int(t)].item())
        return float(np.mean(lps)) if lps else -10.0
    @torch.no_grad()
    def predict_boxes(rec, temperature=0.0):
        new, scores = _generate(rec, temperature)
        boxes = parse_boxes(processor.decode(new, skip_special_tokens=True))
        return boxes, _coord_logprob(new, scores)
    @torch.no_grad()
    def conf_self(rec):
        CONF_PROMPT = ("Look carefully at the liver in this CT image. On a scale of 0 to 100, "
                       "how confident are you that a liver tumor is present? Answer with ONLY one number.")
        new, _ = _generate(rec, prompt=CONF_PROMPT, max_new=8)
        m = re.findall(r"\d+", processor.decode(new, skip_special_tokens=True))
        return min(int(m[0]), 100) / 100.0 if m else 0.5

def _set_iou(A, B):
    """Độ trùng giữa 2 TẬP box (multi-box): tổng matched-IoU / max(số box) — phạt lệch số box."""
    mi, na, nb = match_boxes(A, B)
    return float(np.sum(mi) / max(na, nb)) if mi else (1.0 if (not A and not B) else 0.0)

def conf_spatial(rec, n=None):
    """n lần đoán -> mỗi lần 1 TẬP box -> nhất quán = mean set-IoU từng cặp lần."""
    n = n or globals().get("N_SAMPLES", 5)
    sets = [predict_boxes(rec, temperature=0.7)[0] for _ in range(n)]
    pair = [_set_iou(sets[i], sets[j]) for i in range(len(sets)) for j in range(i + 1, len(sets))]
    return float(np.mean(pair)) if pair else 0.0

print("predict_boxes + conf_logprob(coord)/spatial(set)/self OK | mode:", "MOCK" if USE_MOCK else "REAL")

## PREDICT zero-shot (nhẹ: box + logprob) → JSON

In [ ]:
# === [BASELINE PREDICT NHẸ] Gemma4 ZERO-SHOT đoán test+cal (chỉ box+logprob) -> lưu JSON ===
from tqdm import tqdm
from datetime import datetime
import json as _json, os, torch
torch.manual_seed(globals().get("SEED", 0))            # eval tất định (spatial sample 5 lần)
EVAL_LIMIT = globals().get("EVAL_LIMIT", None)
CAL_LIMIT  = globals().get("CAL_LIMIT", 60)            # số ca cal DƯƠNG để calibrate (None = hết)

def _predict_row(rec):
    boxes, lp = predict_boxes(rec, 0.0)
    return {"pid": rec["patient_id"], "rec": rec,
            "pred_boxes": boxes, "gt_boxes": rec.get("gt_boxes") or [],
            "is_pos": bool(rec.get("gt_boxes")),
            "logprob": lp, "spatial": 0.0, "selfconf": 0.0}   # zero-shot: BO tin hieu nang (signals vo nghia)

_test = dataset["test"][:EVAL_LIMIT] if EVAL_LIMIT else dataset["test"]
_calp = [r for r in dataset["cal"] if r.get("gt_boxes")]
_calp = _calp[:CAL_LIMIT] if CAL_LIMIT else _calp
rows_eval = [_predict_row(r) for r in tqdm(_test, desc="PREDICT test")]
cal_eval  = [_predict_row(r) for r in tqdm(_calp, desc="PREDICT cal+")]

# --- lưu JSON: TÊN DUY NHẤT mỗi lần chạy (timestamp) -> KHÔNG ghi đè lần trước ---
RUN_NAME = globals().get("RUN_NAME", "gemma4_v3")            # tên lần train (từ cfg)
EVAL_DIR = os.path.join(CKPT_DIR, "eval_runs"); os.makedirs(EVAL_DIR, exist_ok=True)
out_json = os.path.join(EVAL_DIR, f"eval_pred_{RUN_NAME}_{datetime.now().strftime('%m%d_%H%M%S')}.json")  # +timestamp -> không ghi đè
def _clean(rows): return [{k: v for k, v in r.items() if k != "rec"} for r in rows]
payload = {"run": RUN_NAME, "n_test": len(rows_eval), "n_cal": len(cal_eval),
           "test": _clean(rows_eval), "cal": _clean(cal_eval)}
with open(out_json, "w") as f:                          # context manager -> flush/đóng đúng (Drive)
    _json.dump(payload, f)
assert os.path.exists(out_json) and os.path.getsize(out_json) > 100, "JSON rỗng / chưa ghi được!"
with open(out_json) as f:                               # đọc lại verify hợp lệ
    _chk = _json.load(f); assert len(_chk["test"]) == len(rows_eval)
print(f"PREDICT xong | test {len(rows_eval)} ({sum(r['is_pos'] for r in rows_eval)} dương) | cal+ {len(cal_eval)}")
print(f"Lưu JSON (Drive, tên riêng lần này): {out_json} ({os.path.getsize(out_json)/1e6:.2f} MB) — verify OK")